# Generating h3 Hexgrids from GeoDataFrames

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from geodatasets import get_path

from tobler.area_weighted import area_interpolate
from tobler.util import h3fy

%load_ext watermark
%watermark -v -a "author: eli knaap" -d -u -p tobler,geopandas

**Note**: This notebook relies on functionality from the [`contextily`](https://contextily.readthedocs.io/en/latest/) package that provides convenient basemaps for geospatial plots. Install the package with  
  
`pip install contextily`     
or    
`conda install contextily -c conda-forge`

In [ ]:
nyc = gpd.read_file(get_path("geoda.nyc_education"))

## Creating Hexgrids with the `h3fy` function

Using the `h3fy` function from the `tobler.util` module, we can easily generate a hexgrid covering the face of the NY.

In [ ]:
nyc_hex = h3fy(nyc, resolution=8)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 10))
axs = axs.flatten()

nyc.plot(ax=axs[0], alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)
nyc_hex.plot(ax=axs[1], alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)

axs[0].set_title("Original Tract Data")
axs[1].set_title("Hex Grid")

for ax in axs.flat:
    ax.axis("off")

By altering the `resolution` parameter, we can generate grids using hexes of various sizes

In [ ]:
nyc_hex_large = h3fy(nyc, resolution=7)
nyc_hex_small = h3fy(nyc, resolution=9)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 10))

nyc_hex_large.plot(ax=axs[0], alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)
nyc_hex_small.plot(ax=axs[1], alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)

for ax in axs.flat:
    ax.axis("off")

and by using the `clip` parameter, we can ensure that the hexgrid is does not extend beyond the borders of the input geodataframe

In [ ]:
nyc_hex_clipped = h3fy(nyc, resolution=7, clip=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

nyc_hex_clipped.plot(ax=ax, alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)
ax.axis("off")
ax.set_title("h3fy with clip")

Notice, though, that we still have some sharp hexagonal edges, meaning the hexagon doesn't *fully* cover the original source polygons. To force full coverage, we can use the `buffer` keyword (or, optionally, both `buffer` and `clip`).

In [ ]:
nyc_hex_clipped = h3fy(nyc, resolution=7, clip=True, buffer=True)

fig, ax = plt.subplots(figsize=(10, 10))

nyc_hex_clipped.plot(ax=ax, alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)
ax.axis("off")
ax.set_title("h3fy with buffer & clip")

## Interpolating to a hexgrid

Thus, in just a few lines of code, we can estimate the value of census variables represented by a regular hexgrid

here, we will estimate the median home value of each hex in the NY using simple areal interpolation

In [ ]:
nyc_hex_interpolated = area_interpolate(
    source_df=nyc, target_df=nyc_hex, intensive_variables=["PER_BLACK"], fill_nan="mean"
)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 10))
nyc.plot("PER_BLACK", scheme="quantiles", alpha=0.7, ax=axs[0], tiles=True)

nyc_hex_interpolated.plot(
    "PER_BLACK",
    scheme="quantiles",
    alpha=0.7,
    ax=axs[1],
    tiles=True,
)

axs[0].set_title("Original Data")
axs[1].set_title("Interpolated Data")

for ax in axs:
    ax.axis("off")
plt.suptitle(
    r"Spatial Interpolation with the PySAL $\mathtt{tobler}$ package", fontsize=16
)

## Dealing with NaN Values

It's important to understand how and NaN (not-a-number) values (i.e. missing data) in the source dataframe are handled. Introduce some missing values.

In [ ]:
nyc.loc[range(0, len(nyc), 25), "PER_BLACK"] = np.nan

In [ ]:
nyc.plot("PER_BLACK", missing_kwds={"color": "red"})

In [ ]:
fill_nans = [None, 0.0, "mean", "median"]

f, ax = plt.subplots(2, 2, figsize=(12, 12))
ax = ax.flatten()
for i, nan in enumerate(fill_nans):
    nyc_hex_interpolated = area_interpolate(
        source_df=nyc,
        target_df=nyc_hex,
        intensive_variables=["PER_BLACK"],
        fill_nan=nan,
    ).plot("PER_BLACK", scheme="quantiles", alpha=0.5, ax=ax[i])
    ax[i].set_title(f"fill_nan={nan}")

Further, all of these options are **still** different from dropping any *geometries* with missing values for the variable being interpolated. In this case, the dropped observations in `source` are not part of the overlay system so only the overlapping pieces of `source` and `target` count toward the total weight applied.

In [ ]:
ax = nyc_hex_interpolated = area_interpolate(
    source_df=nyc.dropna(subset=["PER_BLACK"]),
    target_df=nyc_hex,
    intensive_variables=["PER_BLACK"],
    fill_nan=nan,
).plot("PER_BLACK", scheme="quantiles", alpha=0.5)
ax.set_title("drop NA prior to interpolation")
ax.axis("off")

For extensive variables (where target values are a weighted sum), dropping missing observations is functionally equivalent to setting `fill_nan=0` and `allocate_total=True`. 

For intensive variables, however, the weighted average provided by each `target` zone is not "diluted" by missing data. The total weights at the target polygon is the shared area of source and target, not the total area of target.